# 10. How much information do individual process discriminators retain?


    Compare the **12-bin moving likelihood-ratio observable** from notebook 08
    with fixed one-, two-, and three-dimensional process-ratio histograms.
    All methods use the same analytical Asimov expectation and keep the
    extended-likelihood event count. The analytical unbinned likelihood is the
    full-information benchmark. The comparison asks how much is retained by
    each compression, without assuming that any finite histogram is sufficient.

### Reuse the completed v4 run

Keep **`RUN_NAME = "paper-distinct-s-v4"`**, the same `MODE`, and the same
`CONFIG_OVERRIDES` used for your completed notebooks 01–03 and 08–09. These new
studies do not require regeneration, retraining, or notebooks 04–07. Notebook 10
uses the selected integration bank; notebook 11 also loads the trained ratios
and the **12-bin, a=100** spline model saved by notebook 09.

Start a fresh Colab runtime to load the updated source. The temporary checkout
is `/content/poodemo`; delete that checkout if reusing a runtime with old source,
then restart. Your saved run in Google Drive is separate. The setup downloads
the repository if needed and mounts Drive. For a private repository, give Colab
read access, save a fine-grained token with repository **Contents: read** access
as the `GITHUB_TOKEN` Colab secret, or use the hidden prompt. Alternatively,
upload `/content/poodemo_source.zip`. Tokens are not printed or saved.

`JAX_BACKEND="auto"` uses a Colab GPU when available. Choose `"cpu"` for an
intentional CPU run or `"gpu"` to require a GPU. Setup verifies double-precision
JAX calculations. `MODE="smoke"` uses a separate, already-created smoke run;
changing this notebook's mode does not create the prerequisite samples/models.
Only a migration from an older, incompatible run requires rerunning 01–03.

In [ ]:
import os
import sys
from pathlib import Path

RUN_NAME = "paper-distinct-s-v4"
MODE = os.environ.get("POODEMO_MODE", "production")  # production or smoke
CONFIG_OVERRIDES = {}  # e.g. {"quadrature_per_process": 500_000, "epochs": 150}
JAX_BACKEND = os.environ.get("POODEMO_JAX_BACKEND", "auto")  # auto, gpu, or cpu

try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    import subprocess
    requested_backend = JAX_BACKEND.strip().lower()
    if requested_backend not in ("auto", "gpu", "cpu"):
        raise ValueError("JAX_BACKEND must be auto, gpu, or cpu")
    selected_backend = requested_backend
    if requested_backend == "auto":
        try:
            gpu_probe = subprocess.run(["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"],
                                       capture_output=True, text=True, timeout=10)
            gpu_present = gpu_probe.returncode == 0 and bool(gpu_probe.stdout.strip())
        except (OSError, subprocess.TimeoutExpired):
            gpu_present = False
        selected_backend = "gpu" if gpu_present else "cpu"
    # CUDA-only selection fails visibly if initialization fails. Keep PyTorch's
    # CUDA visibility and avoid JAX reserving most GPU memory before training.
    jax_platform = "cuda" if selected_backend == "gpu" else "cpu"
    os.environ["JAX_PLATFORMS"] = jax_platform
    os.environ.setdefault("XLA_PYTHON_CLIENT_PREALLOCATE", "false")
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
    ROOT = Path(os.environ.get("POODEMO_ROOT", f"/content/drive/MyDrive/poodemo/runs/{RUN_NAME}"))
    REPO_DIR = Path("/content/poodemo")
    if not (REPO_DIR / "pyproject.toml").exists():
        import getpass
        import io
        import shutil
        import tarfile
        import tempfile
        import urllib.error
        import urllib.request
        import zipfile

        uploaded_zip = Path("/content/poodemo_source.zip")
        if uploaded_zip.exists():
            archive_bytes = uploaded_zip.read_bytes()
            archive_kind = "zip"
        else:
            archive_url = "https://api.github.com/repos/rafaellopesdesa/poodemo/tarball/main"

            def fetch_source(token=None):
                headers = {"Accept": "application/vnd.github+json", "User-Agent": "poodemo-colab"}
                if token:
                    headers["Authorization"] = "Bearer " + token
                request = urllib.request.Request(archive_url, headers=headers)
                with urllib.request.urlopen(request, timeout=180) as response:
                    return response.read()

            try:
                archive_bytes = fetch_source()
            except urllib.error.HTTPError as error:
                if error.code not in (401, 403, 404):
                    raise RuntimeError(f"Repository download failed (HTTP {error.code}).") from None
                token = None
                try:
                    from google.colab import userdata
                    token = userdata.get("GITHUB_TOKEN")
                except Exception:
                    pass
                if not token:
                    token = getpass.getpass("GitHub token with read access to poodemo: ").strip()
                if not token:
                    raise RuntimeError("Upload /content/poodemo_source.zip or provide read access.")
                try:
                    archive_bytes = fetch_source(token)
                except urllib.error.HTTPError as auth_error:
                    raise RuntimeError(f"Repository download failed (HTTP {auth_error.code}); check token access.") from None
                finally:
                    token = None
            archive_kind = "tar"

        # Validate archive paths and reject links before extracting the source.
        with tempfile.TemporaryDirectory(prefix="poodemo-source-") as temp:
            staging = Path(temp).resolve()

            def safe_destination(name):
                destination = (staging / name).resolve()
                if not destination.is_relative_to(staging):
                    raise RuntimeError("Unsafe path in source archive.")
                return destination

            if archive_kind == "zip":
                with zipfile.ZipFile(io.BytesIO(archive_bytes)) as archive:
                    for entry in archive.infolist():
                        safe_destination(entry.filename)
                        if (entry.external_attr >> 16) & 0o170000 == 0o120000:
                            raise RuntimeError("Symbolic links are not supported in the source ZIP.")
                    archive.extractall(staging)
            else:
                with tarfile.open(fileobj=io.BytesIO(archive_bytes), mode="r:gz") as archive:
                    for entry in archive.getmembers():
                        safe_destination(entry.name)
                        if not (entry.isfile() or entry.isdir()):
                            raise RuntimeError("Links and special files are not supported in the source archive.")
                    archive.extractall(staging)
            candidates = [p.parent for p in staging.rglob("pyproject.toml") if (p.parent / "poodemo").is_dir()]
            if len(candidates) != 1:
                raise RuntimeError("The uploaded archive must contain one poodemo repository checkout.")
            shutil.copytree(candidates[0], REPO_DIR, dirs_exist_ok=True)
        del archive_bytes

    if os.environ.get("POODEMO_SKIP_INSTALL") != "1":
        from importlib.metadata import version, PackageNotFoundError
        # JAX discovers every installed plugin, even when selecting CPU.
        # Retain already-matched CUDA 12 packages when rerunning GPU setup.
        remove_plugins = []
        for package in ("jax-cuda12-plugin", "jax-cuda12-pjrt", "jax-cuda13-plugin", "jax-cuda13-pjrt"):
            try:
                installed_version = version(package)
            except PackageNotFoundError:
                continue
            if not (selected_backend == "gpu" and package.startswith("jax-cuda12-") and installed_version == "0.5.3"):
                remove_plugins.append(package)
        if remove_plugins:
            subprocess.check_call([sys.executable, "-m", "pip", "uninstall", "-y", "-q", *remove_plugins])
        gpu_requirements = (["jax==0.5.3", "jaxlib==0.5.3",
                             "jax-cuda12-plugin[with-cuda]==0.5.3", "jax-cuda12-pjrt==0.5.3"]
                            if selected_backend == "gpu" else [])
        # Resolve with the base requirements so PyTorch's CUDA dependencies are
        # considered together with JAX's. Avoid blanket --upgrade changes.
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-r",
                               str(REPO_DIR / "requirements-colab.txt"), *gpu_requirements])
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-e", str(REPO_DIR)])

        # Check the live notebook kernel before any expensive network training.
        import jax
        import jaxlib
        import jax.numpy as jnp
        if jax.__version__ != "0.5.3" or jaxlib.__version__ != "0.5.3":
            raise RuntimeError("JAX packages are stale in this kernel. Restart the runtime and rerun setup.")
        if selected_backend == "gpu":
            for package in ("jax-cuda12-plugin", "jax-cuda12-pjrt"):
                if version(package) != "0.5.3":
                    raise RuntimeError(f"{package} must match JAX 0.5.3. Restart the runtime and rerun setup.")
        jax.config.update("jax_platforms", jax_platform)
        jax.config.update("jax_enable_x64", True)
        try:
            jax_devices = jax.devices()
            if not jax_devices or any(device.platform != selected_backend for device in jax_devices):
                raise RuntimeError(f"Expected {selected_backend} devices; found {jax_devices}")
            probe_value, probe_gradient = jax.jit(jax.value_and_grad(lambda x: jnp.sum(x*x)))(
                jnp.asarray([1., 2., 3.], dtype=jnp.float64))
            probe_gradient.block_until_ready()
            if (float(probe_value) != 14. or list(probe_gradient) != [2., 4., 6.]
                    or probe_gradient.dtype != jnp.float64
                    or any(device.platform != selected_backend for device in probe_gradient.devices())):
                raise RuntimeError("JAX double-precision value/gradient check failed")
        except Exception as error:
            raise RuntimeError(
                f"JAX {selected_backend} startup failed. Select a Colab GPU runtime if requesting GPU, "
                "then restart the runtime and rerun setup. Use JAX_BACKEND='cpu' only for an intentional CPU run."
            ) from error
        print(f"JAX {jax.__version__} / jaxlib {jaxlib.__version__}: backend={selected_backend}, {jax_devices}; float64 JIT/gradient OK")

    # A new editable install is not activated in an already-running kernel.
    # Expose the inner package directly, including after a failed import cached
    # the outer /content/poodemo checkout as a namespace package.
    import importlib
    source_root = str(REPO_DIR.resolve())
    if source_root in sys.path:
        sys.path.remove(source_root)
    sys.path.insert(0, source_root)
    importlib.invalidate_caches()
    cached_package = sys.modules.get("poodemo")
    cached_file = getattr(cached_package, "__file__", None)
    expected_init = REPO_DIR / "poodemo" / "__init__.py"
    if cached_package is not None and (cached_file is None or Path(cached_file).resolve() != expected_init.resolve()):
        for module_name in list(sys.modules):
            if module_name == "poodemo" or module_name.startswith("poodemo."):
                del sys.modules[module_name]
else:
    # Locally: pip install -r requirements-colab.txt && pip install -e .
    ROOT = Path(os.environ.get("POODEMO_ROOT", str(Path.home() / "poodemo-runs" / RUN_NAME)))
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        if (candidate / "poodemo" / "pipeline.py").exists():
            sys.path.insert(0, str(candidate))
            break

ROOT.mkdir(parents=True, exist_ok=True)
print(f"Run directory: {ROOT}")
print(f"Mode: {MODE}")

In [ ]:
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from poodemo.pipeline import create_run

run = create_run(ROOT, mode=MODE, overrides=CONFIG_OVERRIDES)
plt.rcParams.update({"figure.figsize": (7, 4.5), "axes.grid": True, "grid.alpha": 0.2})
display(pd.Series(run.config, name="configuration"))


import mplhep as hep
hep.style.use("ATLAS")
# ATLAS-like typography/ticks only: no experiment label is added.
plt.rcParams.update({"axes.grid": False, "figure.dpi": 110})

### Four observable choices

Write \(r_j(x)=p_j(x)/p_S(x)\), with each process density normalized
after the same preselection. For the process axes use
\[
z_j(x)=\frac{r_j(x)^{a_j}}{1+r_j(x)^{a_j}}
=\operatorname{sigmoid}\!\left(a_j\log r_j(x)\right).
\]
A finite positive \(a_j\) preserves ordering while redistributing bin
resolution. The editable starting powers are \(a_{NI}=1\) and
\(a_{SBI}=a_B=2\). Inspect the marginal plots and endpoint-occupation
table before changing them; the powers are fixed for the whole study.

| Observable | Coordinates | Total cells |
|---|---|---:|
| Fixed NI discriminator | \(z_{NI}\) | 12 |
| Fixed NI and SBI discriminators | \((z_{NI},z_{SBI})\) | 144 |
| Fixed NI, SBI and B discriminators | \((z_{NI},z_{SBI},z_B)\) | 1728 |
| Moving physical-reference ratio | \(z_\eta=R_\eta^{100}/(1+R_\eta^{100}),\ R_\eta=p_\eta/p_1\) | 12 |

The default **analytical axes** match the construction in 08–09 and
isolate the information lost by compression and binning. Choosing
`RATIO_SOURCE="learned"` uses trained ratios for both the fixed process
axes and the moving physical-reference axis, adding their approximation
errors. The physical generating
model and analytical benchmark remain unchanged.

For the moving observable, set \(\eta=\mu_0\) at each tested point,
then **freeze the observable and the data** throughout its likelihood
fit. Fixed process axes stay fixed across the entire scan. All use the
same [0,1] edge convention that keeps 0.5 inside a bin. The likelihood
keeps absolute expected counts. Only the marginal display plots are
normalized per process to compare their shapes; fit templates retain yields.

In [ ]:
from poodemo.discriminator_study import run_discriminator_study, plot_discriminator_study

N_BINS = 12  # Per coordinate: 12, 12**2, and 12**3 cells.
POWERS = {"NI": 1.0, "SBI": 2.0, "B": 2.0}
REFERENCE_RATIO_A = 100.0  # Same moving physical-reference ratio as 08/09.
RATIO_SOURCE = "analytic"  # Optional alternative: "learned".
SYSTEMATICS = (False, True)  # Compare both fixed and profiled NI.
MORPH_ORDER = "after"  # Interpolate the binned NI templates, as in 09.
MU_GRID = None  # None uses the saved run scan; e.g. np.linspace(0.02, 2., 101).

print(f"Fixed grids: {N_BINS}, {N_BINS**2}, {N_BINS**3} cells")
print("Process powers:", POWERS, "| physical-reference power:", REFERENCE_RATIO_A)
print("Observable axes:", RATIO_SOURCE, "| Asimov truth:", run.config["asimov_mu"])

### Build the templates and scan the likelihood

The physical \(\mu\) coefficients and selected process yields are
identical across the histograms. Stat-only fixes the NI nuisance at
zero; the profiled comparison includes its Gaussian constraint and
the binned nuisance interpolation used in notebook 09. The diagnostic
`MORPH_ORDER` setting can separately investigate interpolation versus
integration. Default settings do not change the saved physics run.

The occupancy table reports sparse and empty cells, together with
integration-bank effective sample sizes. Going from 12 to 1728 cells
increases the integration demands, even if the physical information
improves. Sparse cells are reported, not filled with artificial events.

In [ ]:
study = run_discriminator_study(
    run, n_bins=N_BINS, powers=POWERS, reference_power=REFERENCE_RATIO_A,
    ratio_source=RATIO_SOURCE, systematics=SYSTEMATICS,
    scan_grid=MU_GRID, morph_order=MORPH_ORDER,
)
display(study["axis_summary"])
display(study["occupancy"])
print("Full settings:")
display(pd.Series(study["config"]))

### Distributions, likelihood scans, and local information

First inspect the one-dimensional axes and their joint distributions.
Then compare the likelihood scans directly. The information panel
measures **local precision at the fixed Asimov truth**, which is a
different question from retaining a finite-separation test against 1.
At \(\eta=1\), the moving observable collapses to 0.5 and contains only
event-count information. Its test of the generating truth can still
correctly give zero even though its frozen local width is poor.

In [ ]:
FIGURE_DIR = ROOT / "plots" / "10_discriminator_study"
figures = plot_discriminator_study(study, output=FIGURE_DIR)
for name, figure in figures.items():
    print(name)
    display(figure)
    plt.close(figure)
print("Figures:", FIGURE_DIR)
print("Tables:", ROOT / "results" / "10_discriminator_study")

In [ ]:
display(study["information"])
scans = study["scans"]
if "valid" in scans.columns:
    invalid = scans.loc[~scans["valid"].fillna(False)]
    if len(invalid):
        print("Inspect these unsuccessful fits before interpreting the curves:")
        display(invalid)
display(scans.head(15))

### What this demonstrates

With NI fixed, the nominal intensity is a known linear combination of
S, SBI, B and NI. Consequently the **continuous vector**
\((r_{NI},r_{SBI},r_B)\), together with the event count, contains the
information needed for the nominal \(\mu\) family: the common S density
cancels from likelihood ratios. Finite 12-by-12-by-12 bins still lose
information, and sparse integration cells can limit the comparison.
When NI is profiled, its variation ratios need not be functions of this
nominal three-ratio vector, so this sufficiency argument is narrower.

SBI and B are similar in this model, but that alone does not establish
that the third axis is redundant: their difference helps reconstruct
the interference contribution. Measure its incremental gain from the
actual fits. The fixed grids are nested coordinate refinements, but
this does not require every plotted profiled test statistic to increase
monotonically. The comparison with the moving 12-bin ratio shows how efficiently a
test-specific ordering can use a small bin budget. This does not prove
that one fixed scalar retains the entire parameter family, nor that
the profiled or learned cases inherit exact pairwise optimality.

Next: **11_test_statistic_toys.ipynb**, which replaces Asimov
expectations with repeated experiments and measures calibration.